# 09 — Final comparison: all models, all protocols, with confidence intervals

Loads each fine-tuned checkpoint, generates on the **same** 1,000-example test set, scores every
model under all four ROUGE protocols, and produces the four tables the paper needs. Nothing is
transcribed by hand -- every number comes from this run.

**What it adds beyond the per-model notebooks:** bootstrap confidence intervals on the
*differences* between models. AraT5 currently leads AraBART by 0.27 ROUGE-1, which is almost
certainly not a real difference. Without an interval there is no way to say so defensibly, and
"model A scored 0.27 higher" is exactly the kind of claim reviewers reject.

Runtime is roughly 15-20 minutes, nearly all of it generation. No training.

## Step 0 — Setup

Run the pin cell, restart the kernel, then run from the top. `transformers` 5.x broke
SentencePiece tokenizer loading for both AraBART (`TypeError: 'dict' object cannot be converted
to 'Sequence'`) and AraT5 (`KeyError: 0`). 4.47.1 loads all three models correctly; weights are
unaffected.

In [ ]:
!pip install -q "transformers==4.47.1"
# RESTART THE KERNEL after this cell, then run from the top.


In [ ]:
!pip install -q evaluate rouge_score bert_score absl-py nltk qalsadi


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re, gc, json, time, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
import transformers
from nltk.stem import SnowballStemmer
from rouge_score import rouge_scorer
import qalsadi.lemmatizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig
from tqdm import tqdm

nltk.download("stopwords", quiet=True)
print("transformers:", transformers.__version__)
if transformers.__version__.startswith("5."):
    print("WARNING: still on 5.x -- AraBART and AraT5 tokenizers will fail. Run the pin cell "
          "above and restart the kernel.")

DATA_DIR = "/kaggle/input/datasets/mohamedbentalb/arabic-summ-canonical-v1/canonical_dataset"
# ---- The models to compare. `base` is the tokenizer fallback and MUST match the checkpoint's
# ---- architecture: another model's tokenizer emits ids past the embedding table, which shows
# ---- up as an unrelated CUDA "device-side assert".
# ============================  RUN MODE  ============================
# "finetuned" -> evaluate your trained checkpoints   -> /kaggle/working/comparison
# "zeroshot"  -> evaluate the untrained base models  -> /kaggle/working/comparison_zeroshot
# The two write to different directories, so a zero-shot run can never overwrite the
# fine-tuned results (or the reverse).
RUN_MODE = "zeroshot"
# ====================================================================

FINETUNED_MODELS = {
    # mT5 MUST point at the Trainer checkpoint, not the *_finetuned folder: that folder was
    # written after a tie_weights() call which overwrote the trained lm_head, so it generates
    # multilingual noise. Fallback if needed: checkpoint-23735 (epoch 5, verified good).
    "mT5-small": {
        "path": "/kaggle/input/models/medben07/t5-finetuned-last-checkpoint/transformers/default/1/results/checkpoints/checkpoint-37976",
        "base": "google/mt5-small", "params_m": 300, "best_epoch": 8},
    "AraBART": {
        "path": "/kaggle/input/models/medben07/fine-tuned-arabart/transformers/default/1/results/arabart_finetuned",
        "base": "moussaKam/AraBART", "params_m": 139, "best_epoch": 4},
    "AraT5v2-base": {
        "path": "/kaggle/input/models/mb032000/finetuned-arat5/transformers/default/1/results/arat5_finetuned",
        "base": "UBC-NLP/AraT5v2-base-1024", "params_m": 368, "best_epoch": 4},
}

# Untrained models straight from the Hub, for the zero-shot baseline column.
# mT5 is expected to FAIL the Arabic sanity gate: T5 span-corruption pretraining emits
# sentinel tokens rather than fluent text. That failure is a reportable result, not a bug --
# keep SKIP_BROKEN_MODELS = True so the other two still run.
BASE_MODELS = {
    "mT5-small":    {"path": "google/mt5-small",          "base": "google/mt5-small",
                     "params_m": 300, "best_epoch": 0},
    "AraBART":      {"path": "moussaKam/AraBART",         "base": "moussaKam/AraBART",
                     "params_m": 139, "best_epoch": 0},
    "AraT5v2-base": {"path": "UBC-NLP/AraT5v2-base-1024", "base": "UBC-NLP/AraT5v2-base-1024",
                     "params_m": 368, "best_epoch": 0},
}

assert RUN_MODE in ("finetuned", "zeroshot"), f"unknown RUN_MODE: {RUN_MODE}"
MODELS = FINETUNED_MODELS if RUN_MODE == "finetuned" else BASE_MODELS
RESULTS_DIR = ("/kaggle/working/comparison" if RUN_MODE == "finetuned"
               else "/kaggle/working/comparison_zeroshot")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"RUN_MODE = {RUN_MODE}  ->  results go to {RESULTS_DIR}")

# Set True to keep going when a model fails its checks, instead of aborting the whole run.
SKIP_BROKEN_MODELS = True

MAX_INPUT_LEN, GEN_MAX_LEN, NUM_BEAMS, BATCH_SIZE = 512, 150, 4, 8
N_BOOTSTRAP = 2000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

if torch.cuda.is_available():
    _cc = torch.cuda.get_device_capability(0); _sm = f"sm_{_cc[0]}{_cc[1]}"
    assert _sm in torch.cuda.get_arch_list(), (
        f"GPU is {_sm} but this PyTorch build has no kernels for it. On Kaggle, select T4 "
        "(P100 is sm_60 and the default image is built for sm_70+).")
    _t = (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
    torch.cuda.synchronize(); torch.cuda.empty_cache()
    print(f"CUDA OK ({_sm}, smoke test {_t:.1f})")


def is_hub_id(p):
    """True for a HuggingFace Hub id such as "google/mt5-small", not a local directory."""
    return (not p.startswith("/")) and (not os.path.isdir(p)) and p.count("/") == 1


def find_path(expected, marker="config.json"):
    """Return `expected` if usable, else search /kaggle/input and report candidates.

    Hub ids pass straight through: from_pretrained resolves them by download, and
    os.path.exists is always False for them.
    """
    if is_hub_id(expected):
        return expected
    if os.path.exists(os.path.join(expected, marker)):
        return expected
    print(f"  NOT FOUND: {expected}")
    # Include checkpoint-* folders: when a *_finetuned folder is unusable (e.g. its output head
    # was overwritten by tie_weights after training), the Trainer checkpoint written earlier in
    # the same run is the correct source.
    hits = [d for d, _, f in os.walk("/kaggle/input") if marker in f]
    for h in sorted(hits):
        tag = "  <- Trainer checkpoint" if "checkpoint-" in h else ""
        print(f"    candidate: {h}{tag}")
    return None


print("\nResolving model paths:")
for k, cfg in MODELS.items():
    p = find_path(cfg["path"])
    cfg["resolved"] = p
    tag = ("OK (Hub, will download)" if (p and is_hub_id(p))
           else ("OK" if p else "MISSING -- fix MODELS['" + k + "']['path'] above"))
    print(f"  {k}: {tag}")


## Step 1 — Test set

In [ ]:
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
articles = test_df["article"].tolist()
references = test_df["summary"].tolist()
sources = test_df["source"].tolist() if "source" in test_df.columns else ["all"] * len(test_df)
print(f"{len(test_df)} test examples | by source: {pd.Series(sources).value_counts().to_dict()}")

SLICES = {"ALL": list(range(len(references)))}
for s in sorted(set(sources)):
    SLICES[s] = [i for i, v in enumerate(sources) if v == s]


## Step 2 — Scoring protocols

Four tokenizers sharing one scorer, so normalisation is the only variable. Per-example scores
are kept (not just the mean) because the bootstrap in Step 6 resamples them.

In [ ]:
_PUNCT_RE = re.compile(r"([^\w\s])", flags=re.UNICODE)


def _basic_tokenize(text):
    text = text.lower()
    return [t for t in _PUNCT_RE.sub(r" \1 ", text).split() if t]


class SplitTokenizer:
    name = "whitespace"
    def tokenize(self, text): return text.split()


class PunctTokenizer:
    name = "punct-aware"
    def tokenize(self, text): return _basic_tokenize(text)


class StemmedTokenizer:
    name = "punct-aware + stemmed"
    def __init__(self): self.s = SnowballStemmer("arabic", ignore_stopwords=True)
    def tokenize(self, text): return [self.s.stem(t) for t in _basic_tokenize(text)]


class LemmaTokenizer:
    name = "punct-aware + lemmatized"
    def __init__(self):
        self.l = qalsadi.lemmatizer.Lemmatizer(); self.c = {}
    def _lem(self, w):
        if w not in self.c:
            try: self.c[w] = self.l.lemmatize(w) or w
            except Exception: self.c[w] = w
        return self.c[w]
    def tokenize(self, text): return [self._lem(t) for t in _basic_tokenize(text)]


PROTOCOLS = [SplitTokenizer(), PunctTokenizer(), StemmedTokenizer(), LemmaTokenizer()]
KEYS = ["rouge1", "rouge2", "rougeL"]


def per_example_rouge(preds, refs, tok):
    """Per-example f-measures, so the bootstrap can resample without re-scoring."""
    sc = rouge_scorer.RougeScorer(KEYS, use_stemmer=False, tokenizer=tok)
    out = {k: np.zeros(len(preds)) for k in KEYS}
    for i, (p, r) in enumerate(zip(preds, refs)):
        s = sc.score(r, p)
        for k in KEYS: out[k][i] = s[k].fmeasure
    return out


_p = "أعلن متحدث باسم فرق الإطفاء أنّ الحريق الضخم اندلع في كاتدرائية نوتردام"
_r = "أعلنت فرق الإطفاء أنّ الحريق الضخم الذي اندلع في كاتدرائية نوتردام في باريس"
print("Self-test (all > 0):")
for t in PROTOCOLS:
    v = per_example_rouge([_p], [_r], t)["rouge1"][0]
    print(f"  {t.name:<26} {v:.4f}")
    assert v > 0, f"{t.name} scored zero on matching Arabic -- tokenizer broken"


def arabic_ratio(text):
    if not text or not text.strip(): return 0.0
    return sum(1 for ch in text if "\u0600" <= ch <= "\u06FF") / len(text)


## Step 3 — Generate for every model

One model at a time, freed before the next so peak memory stays at a single model. Each is
vocabulary-checked on CPU and sanity-probed before the full pass.

In [ ]:
def inspect_checkpoint_file(path):
    """Report on lm_head/shared in the saved file itself, before any loading.

    Two distinct failure modes both destroy a fine-tuned T5-family output head:
      - lm_head.weight present but identical to shared.weight -> tie_weights() ran before saving
      - lm_head.weight absent entirely -> save_pretrained dropped it as 'tied', and reload
        rebuilds it from shared, silently discarding what the head learned
    """
    if is_hub_id(path):
        return ("hub", "Hub id -- loaded as published, no local file to inspect")
    if not os.path.isdir(path):
        return None
    try:
        from safetensors import safe_open
    except ImportError:
        return None
    st = [f for f in os.listdir(path) if f.endswith(".safetensors")]
    if not st:
        return None
    with safe_open(os.path.join(path, st[0]), framework="pt") as f:
        keys = set(f.keys())
        if "lm_head.weight" not in keys:
            return ("absent", "lm_head.weight is not in the file: save_pretrained dropped it as "
                              "tied, so reload rebuilds it from shared and the trained head is lost")
        if "shared.weight" not in keys:
            return ("ok", "lm_head.weight present (no shared.weight to compare)")
        same = torch.equal(f.get_tensor("lm_head.weight"), f.get_tensor("shared.weight"))
        return (("tied", "lm_head.weight is byte-identical to shared.weight in the file")
                if same else ("ok", "lm_head.weight differs from shared.weight in the file"))


def load_checked(path, base):
    """Load a checkpoint, verifying head, tokenizer and vocabulary before touching the GPU.

    Two architectures behave differently and both are legitimate:
      - BART (AraBART) ties lm_head to the shared embedding *by design*; lm_head.weight is in
        its _tied_weights_keys and is correctly absent from the saved file.
      - T5-family (mT5, AraT5) keep a separate lm_head that diverges during training. If the
        file holds a distinct lm_head but config.tie_word_embeddings is True, transformers 4.x
        re-ties on load and silently discards the trained head, so tying is disabled explicitly.
    """
    status, msg = inspect_checkpoint_file(path) or ("unknown", "no safetensors file to inspect")
    print(f"    file check: {msg}")
    if status == "tied":
        raise RuntimeError(
            f"checkpoint at {path} has lm_head byte-identical to shared in the file -- "
            "tie_weights() ran before saving and the trained head is gone. Use a Trainer "
            "checkpoint-* folder written before that step.")

    cfg = AutoConfig.from_pretrained(path)
    if status == "ok" and getattr(cfg, "tie_word_embeddings", False):
        cfg.tie_word_embeddings = False
        print("    config says tie_word_embeddings=True but the file has a distinct lm_head;")
        print("    loading with tying disabled so the trained head survives")
    model = AutoModelForSeq2SeqLM.from_pretrained(path, config=cfg)

    try:
        tok = AutoTokenizer.from_pretrained(path)
        src = "checkpoint"
    except Exception as e:
        print(f"    checkpoint tokenizer failed ({type(e).__name__}: {str(e)[:70]}); using {base}")
        tok = AutoTokenizer.from_pretrained(base)
        src = f"fallback {base}"

    rows = model.get_input_embeddings().weight.shape[0]
    assert len(tok) <= rows, (
        f"tokenizer ({len(tok)}) exceeds embedding rows ({rows}) -- wrong tokenizer for this "
        "checkpoint; ids would index out of bounds and CUDA would raise a device-side assert")
    print(f"    tokenizer from {src}; vocab {len(tok)} <= {rows} embedding rows")

    # Only meaningful when the file carried a distinct head. For an architecturally tied model
    # (BART) identical weights are correct, so this check does not apply.
    _lm, _sh = getattr(model, "lm_head", None), getattr(model, "shared", None)
    if status == "ok" and _lm is not None and _sh is not None and hasattr(_lm, "weight"):
        if torch.equal(_lm.weight.detach().cpu(), _sh.weight.detach().cpu()):
            raise RuntimeError(
                "the file had a distinct lm_head but the loaded model has it tied to shared -- "
                "disabling tie_word_embeddings did not take effect.")
        print("    lm_head differs from shared after load (trained head preserved)")
    elif status == "absent":
        print("    lm_head tied to shared by architecture (normal for BART) -- "
              "the sanity probe below is the real test")

    return model.to(device).eval(), tok


def generate(model, tok, arts, bs=BATCH_SIZE):
    preds = []
    for i in tqdm(range(0, len(arts), bs), desc="    generating", leave=False):
        enc = tok(arts[i:i+bs], return_tensors="pt", truncation=True,
                  padding=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_length=GEN_MAX_LEN,
                                 num_beams=NUM_BEAMS, early_stopping=True)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds


PREDICTIONS, GEN_TIME, FAILURES = {}, {}, {}
for name, cfg in MODELS.items():
    print()
    print(f"{name}")
    print(f"    path: {cfg['path']}")
    if cfg["resolved"] is None:
        FAILURES[name] = "path unresolved"
        print("    SKIPPED (path unresolved)")
        continue
    try:
        model, tok = load_checked(cfg["resolved"], cfg["base"])

        probe = generate(model, tok, articles[:3])
        ratios = [arabic_ratio(p) for p in probe]
        if np.mean(ratios) <= 0.5:
            raise RuntimeError(
                f"output is not Arabic (mean ratio {np.mean(ratios):.2f}). Sample: {probe[0][:100]}")
        print(f"    sanity probe OK (Arabic ratio {np.mean(ratios):.2f})")

        t0 = time.time()
        PREDICTIONS[name] = generate(model, tok, articles)
        GEN_TIME[name] = time.time() - t0
        empty = sum(1 for p in PREDICTIONS[name] if not p.strip())
        print(f"    {len(PREDICTIONS[name])} summaries in {GEN_TIME[name]:.0f}s | empty: {empty}")

        pd.DataFrame({"article": articles, "reference": references,
                      "prediction": PREDICTIONS[name], "source": sources}
                     ).to_csv(f"{RESULTS_DIR}/{name}_predictions.csv", index=False)
    except Exception as e:
        FAILURES[name] = f"{type(e).__name__}: {e}"
        print(f"    FAILED: {type(e).__name__}: {str(e)[:200]}")
        if not SKIP_BROKEN_MODELS:
            raise
    finally:
        for _v in ("model", "tok"):
            if _v in dir():
                exec(f"del {_v}")
        gc.collect(); torch.cuda.empty_cache()

print()
print("=" * 70)
print(f"Generated for : {list(PREDICTIONS)}")
if FAILURES:
    print("Failed        :")
    for k, v in FAILURES.items():
        print(f"  {k}: {v[:160]}")
assert PREDICTIONS, "no model produced predictions -- fix the paths above before continuing"


## Step 4 — Per-example scores

Computed once per model x protocol and reused by every table and by the bootstrap.

In [ ]:
SCORES = {}   # SCORES[model][protocol][metric] -> np.array over test examples
for name, preds in PREDICTIONS.items():
    SCORES[name] = {}
    for tok in tqdm(PROTOCOLS, desc=name, leave=False):
        SCORES[name][tok.name] = per_example_rouge(preds, references, tok)
print("scored:", {m: list(p) for m, p in SCORES.items()})


def mean_score(model, protocol, metric, idx=None):
    a = SCORES[model][protocol][metric]
    return float(np.mean(a if idx is None else a[idx])) * 100


## Step 5 — BERTScore

Computed once per model over all examples, then averaged per slice. Unaffected by the
tokenisation issues above, so it is an independent check on the ROUGE ordering.

In [ ]:
bertscore = evaluate.load("bertscore")
BERT = {}
for name, preds in PREDICTIONS.items():
    r = bertscore.compute(predictions=preds, references=references, lang="ar")
    BERT[name] = {"f1": np.array(r["f1"]), "precision": np.array(r["precision"]),
                  "recall": np.array(r["recall"])}
    print(f"{name}: BERTScore-F1 {np.mean(r['f1']):.4f}")


## Table 1 — Main controlled comparison

In [ ]:
PRIMARY = "punct-aware + lemmatized"

rows = []
for name in PREDICTIONS:
    rows.append({
        "Model": name,
        "Params (M)": MODELS[name]["params_m"],
        "Best epoch": MODELS[name]["best_epoch"],
        "ROUGE-1": mean_score(name, PRIMARY, "rouge1"),
        "ROUGE-2": mean_score(name, PRIMARY, "rouge2"),
        "ROUGE-L": mean_score(name, PRIMARY, "rougeL"),
        "BERTScore-F1": float(np.mean(BERT[name]["f1"])) * 100,
    })
table1 = pd.DataFrame(rows).set_index("Model").round(2)
print(f"Protocol: {PRIMARY} | n = {len(references)}\n")
print(table1.to_string())
print("\n" + table1.to_markdown())


## Table 2 — Protocol sensitivity

In [ ]:
rows = []
for tok in PROTOCOLS:
    r = {"Protocol": tok.name}
    for name in PREDICTIONS:
        r[name] = mean_score(name, tok.name, "rouge1")
    rows.append(r)
table2 = pd.DataFrame(rows).set_index("Protocol").round(2)

swing = {n: (table2.loc[PRIMARY, n] / table2.loc["whitespace", n] - 1) * 100 for n in PREDICTIONS}
table2.loc["relative swing (%)"] = pd.Series(swing).round(1)
print("ROUGE-1 under each scoring protocol (same model, same outputs, same test set)\n")
print(table2.to_string())
print("\n" + table2.to_markdown())


## Table 3 — Domain breakdown

In [ ]:
rows = []
for name in PREDICTIONS:
    r = {"Model": name}
    for s, idx in SLICES.items():
        r[f"{s} (n={len(idx)})"] = mean_score(name, PRIMARY, "rouge1", idx)
    rows.append(r)
table3 = pd.DataFrame(rows).set_index("Model").round(2)
print(f"ROUGE-1 by source, protocol = {PRIMARY}\n")
print(table3.to_string())
print("\n" + table3.to_markdown())


## Table 4 — Prior work (indicative only)

Kahla et al. (2023) evaluated on the official AraSum split (~4,000 examples) with stemmed ROUGE,
after ~10x more gradient steps on 8 GPUs. Our nearest-comparable cell is the arasum slice under
the stemmed protocol. **Do not merge these rows into Table 1** -- different test set, different
training budget.

In [ ]:
KAHLA = {
    "XL-Sum baseline":           (30.026, 12.874, 23.836),
    "mT5 (AraSum only)":         (32.859, 13.843, 24.571),
    "mT5++ (AraSum + XL-Sum)":   (33.172, 13.914, 24.782),
    "mBART-50-rus (their best)": (33.842, 16.049, 26.531),
}
STEM = "punct-aware + stemmed"
ara = SLICES.get("arasum")

rows = [{"System": k, "ROUGE-1": v[0], "ROUGE-2": v[1], "ROUGE-L": v[2],
         "Source": "Kahla et al. (2023)"} for k, v in KAHLA.items()]
if ara is not None:
    for name in PREDICTIONS:
        rows.append({"System": f"{name} (ours)",
                     "ROUGE-1": mean_score(name, STEM, "rouge1", ara),
                     "ROUGE-2": mean_score(name, STEM, "rouge2", ara),
                     "ROUGE-L": mean_score(name, STEM, "rougeL", ara),
                     "Source": f"this work, arasum slice n={len(ara)}"})
table4 = pd.DataFrame(rows).set_index("System").round(2)
print(f"Stemmed ROUGE, to match their protocol\n")
print(table4.to_string())
print("\n" + table4.to_markdown())


## Table 5 — Extractiveness

How much of each summary is copied verbatim from its source article, using the
extractive-fragment measures of Grusky et al. (2018).

* **coverage** — fraction of summary tokens found somewhere in the article.
* **density** — mean squared extractive-fragment length. A density near 1 means the
  shared words are isolated; a high density means long verbatim spans were copied.
* **novel n-gram rate** — share of summary n-grams absent from the article. Higher is
  more abstractive.

The reference summaries are scored too, and that row is the important one: it sets the
level of abstraction the task actually calls for. This measure compares each summary
against its **source article**, unlike ROUGE and BERTScore, which compare it against the
**reference summary** — so it answers a question those metrics cannot.

In [ ]:
from tqdm import tqdm as _tqdm


def _frag_tokens(text):
    return [t for t in _PUNCT_RE.sub(r" \1 ", str(text).lower()).split() if t]


def extractive_fragments(article_toks, summary_toks):
    """Greedy longest-match fragments shared by article and summary (Grusky et al., 2018)."""
    frags, i, j = [], 0, 0
    while i < len(summary_toks):
        best = []
        while j < len(article_toks):
            if summary_toks[i] == article_toks[j]:
                i_, j_ = i, j
                while (i_ < len(summary_toks) and j_ < len(article_toks)
                       and summary_toks[i_] == article_toks[j_]):
                    i_ += 1; j_ += 1
                if len(best) < i_ - i:
                    best = summary_toks[i:i_]
                j = j_
            else:
                j += 1
        i, j = i + max(len(best), 1), 0
        if best:
            frags.append(best)
    return frags


def novel_ngram_rate(article_toks, summary_toks, n):
    if len(summary_toks) < n:
        return np.nan
    seen = {tuple(article_toks[k:k+n]) for k in range(len(article_toks) - n + 1)}
    grams = [tuple(summary_toks[k:k+n]) for k in range(len(summary_toks) - n + 1)]
    return sum(1 for g in grams if g not in seen) / len(grams)


# Tokenise the articles once; this is the expensive part.
_article_toks = [_frag_tokens(a) for a in articles]

# The gold references are included so the models can be read against the level of
# abstraction the task itself requires.
_systems = {"reference (gold)": references}
_systems.update(PREDICTIONS)

rows = []
for name, summaries in _systems.items():
    cov, den, n1, n2, n3 = [], [], [], [], []
    for art, summ in _tqdm(list(zip(_article_toks, summaries)), desc=name, leave=False):
        st = _frag_tokens(summ)
        if not st:
            continue
        frags = extractive_fragments(art, st)
        total = sum(len(f) for f in frags)
        cov.append(total / len(st))
        den.append(sum(len(f) ** 2 for f in frags) / len(st))
        n1.append(novel_ngram_rate(art, st, 1))
        n2.append(novel_ngram_rate(art, st, 2))
        n3.append(novel_ngram_rate(art, st, 3))
    rows.append({"System": name,
                 "Coverage": float(np.mean(cov)),
                 "Density": float(np.mean(den)),
                 "Novel 1-gram (%)": float(np.nanmean(n1)) * 100,
                 "Novel 2-gram (%)": float(np.nanmean(n2)) * 100,
                 "Novel 3-gram (%)": float(np.nanmean(n3)) * 100})

table5 = pd.DataFrame(rows).set_index("System").round(3)
print(table5.to_string())
print()
print(table5.to_markdown())

_ref = table5.loc["reference (gold)"]
print(f"\nReference density is {_ref['Density']:.2f}; every model above that value is "
      f"copying longer verbatim spans than the task requires.")
for m in [k for k in table5.index if k != "reference (gold)"]:
    r = table5.loc[m]
    print(f"  {m:<14} density {r['Density']:.2f} "
          f"({r['Density'] / _ref['Density']:.1f}x reference), "
          f"novel 3-grams {r['Novel 3-gram (%)']:.1f}% vs {_ref['Novel 3-gram (%)']:.1f}%")


## Step 6 — Bootstrap confidence intervals

Resamples the 1,000 test examples with replacement `N_BOOTSTRAP` times and recomputes each
model's mean from the stored per-example scores -- no re-scoring, so it takes seconds.

Read the **difference** intervals: if a 95% CI straddles zero, the models are not
distinguishable on this test set and should be reported as comparable rather than ranked.

In [ ]:
n = len(references)
rng = np.random.default_rng(RANDOM_SEED)
boot_idx = rng.integers(0, n, size=(N_BOOTSTRAP, n))

def boot_means(name, metric, protocol=PRIMARY):
    a = SCORES[name][protocol][metric]
    return a[boot_idx].mean(axis=1) * 100

print(f"Per-model 95% CI ({N_BOOTSTRAP} resamples, protocol = {PRIMARY})\n")
print(f"{'Model':<16}{'metric':<10}{'mean':>8}{'95% CI':>22}")
ci_rows = []
for name in PREDICTIONS:
    for m in KEYS:
        d = boot_means(name, m)
        lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"{name:<16}{m:<10}{d.mean():>8.2f}   [{lo:>6.2f}, {hi:>6.2f}]")
        ci_rows.append({"model": name, "metric": m, "mean": d.mean(), "ci_low": lo, "ci_high": hi})

print(f"\n\nPairwise differences, 95% CI on (A - B)\n")
print(f"{'comparison':<34}{'metric':<10}{'diff':>8}{'95% CI':>22}  verdict")
names = list(PREDICTIONS)
diff_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        for m in KEYS:
            d = boot_means(a, m) - boot_means(b, m)
            lo, hi = np.percentile(d, [2.5, 97.5])
            sig = "DISTINGUISHABLE" if (lo > 0 or hi < 0) else "not distinguishable"
            print(f"{a + ' - ' + b:<34}{m:<10}{d.mean():>+8.2f}   [{lo:>+6.2f}, {hi:>+6.2f}]  {sig}")
            diff_rows.append({"a": a, "b": b, "metric": m, "diff": d.mean(),
                              "ci_low": lo, "ci_high": hi, "significant": bool(lo > 0 or hi < 0)})

ci_df, diff_df = pd.DataFrame(ci_rows), pd.DataFrame(diff_rows)


## Step 7 — Save everything

In [ ]:
for nm, df in [("table1_main", table1), ("table2_protocol", table2),
               ("table3_domain", table3), ("table4_prior_work", table4),
               ("table5_extractiveness", table5),
               ("bootstrap_ci", ci_df), ("bootstrap_diffs", diff_df)]:
    df.to_csv(f"{RESULTS_DIR}/{nm}.csv")

payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "n_test_examples": len(references),
    "primary_protocol": PRIMARY,
    "n_bootstrap": N_BOOTSTRAP,
    "models": {k: {kk: vv for kk, vv in v.items() if kk != "resolved"} for k, v in MODELS.items()},
    "generation_time_s": GEN_TIME,
    "table1_main": json.loads(table1.reset_index().to_json(orient="records")),
    "table2_protocol": json.loads(table2.reset_index().to_json(orient="records")),
    "table3_domain": json.loads(table3.reset_index().to_json(orient="records")),
    "table4_prior_work": json.loads(table4.reset_index().to_json(orient="records")),
    "table5_extractiveness": json.loads(table5.reset_index().to_json(orient="records")),
    "bootstrap_ci": json.loads(ci_df.to_json(orient="records")),
    "bootstrap_diffs": json.loads(diff_df.to_json(orient="records")),
    "notes": {
        "rouge": "rouge_score with Unicode-safe tokenizers; the stock default empties Arabic text",
        "stemmed": "SnowballStemmer('arabic', ignore_stopwords=True), matching Kahla et al. / XL-Sum",
        "prior_work": "different test split and ~10x training budget -- indicative, not a benchmark result",
    },
}
with open(f"{RESULTS_DIR}/final_comparison.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:")
for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {fn}")


## Reading the output

**Table 1** is the headline. Report the bootstrap CIs alongside it -- if the AraBART/AraT5
difference straddles zero, say the two are comparable rather than ranking them.

**Table 2** is the evaluation-critique contribution: the same model and outputs scored four ways.

**Table 4** must stay labelled indicative. Merging it into Table 1 would be exactly the
confounded comparison this project set out to criticise.